# Part 1: Neural Network Analysis - Customer Churn Prediction

This notebook trains a feed-forward neural network on a customer churn dataset using **scikit-learn MLPClassifier**.

**Dataset:** `customer_churn_nn.csv` - 2000 rows, 17 columns, binary target `churn`

## Task 1: Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../datasets/ai_project_synthetic_datasets/part_1_neural_network_analysis/customer_churn_nn.csv')
print("Shape (rows, columns):", df.shape)
print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Column data types - understanding input features
print("Column Data Types:")
print(df.dtypes)
print("\nCategorical columns:", df.select_dtypes(include='object').columns.tolist())
print("Numerical columns:", df.select_dtypes(include='number').columns.tolist())

In [ ]:
# Missing value check
print("Missing Values per Column:")
missing = df.isnull().sum()
print(missing)
print("\nTotal missing values:", missing.sum())

In [ ]:
# Statistical summary
print("Basic Statistical Summary:")
df.describe()

In [ ]:
# Target variable description and distribution
print("Target Variable: 'churn' (0 = No Churn, 1 = Churned)")
print("\nValue Counts:")
print(df['churn'].value_counts())
print("\nPercentage:")
print(df['churn'].value_counts(normalize=True).mul(100).round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['churn'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Churn Distribution (Count)')
axes[0].set_xticklabels(['No Churn (0)','Churn (1)'], rotation=0)
axes[0].set_ylabel('Count')
df['churn'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue','tomato'], startangle=90)
axes[1].set_title('Churn Distribution (%)')
axes[1].set_ylabel('')
plt.suptitle('Distribution of Target Variable: Churn')
plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=120)
plt.show()

## Task 2: Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Step 1: Drop non-informative ID column
df2 = df.drop(columns=['customer_id'])

# Step 2: Encode categorical columns
# Label Encoding converts text categories to numbers (e.g., 'North'->0, 'South'->1)
cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
le = LabelEncoder()
for col in cat_cols:
    df2[col] = le.fit_transform(df2[col])
    print(f"Encoded '{col}': unique values -> {df2[col].unique()}")

# Step 3: Separate features (X) and target (y)
X = df2.drop(columns=['churn'])
y = df2['churn']
print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Step 4: Scale features using StandardScaler
# Neural networks train better when all features are on the same scale (mean=0, std=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 5: Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"Features per sample: {X_train.shape[1]}")

## Task 3: Neural Network Model Building

> We use **`MLPClassifier`** from scikit-learn, which implements a multi-layer perceptron (feed-forward neural network).

**Architecture:**
- **Input layer:** 15 neurons (one per feature)
- **Hidden layer:** 64 neurons, ReLU activation
- **Output layer:** 1 neuron (sigmoid for binary classification)
- **Loss function:** Binary Cross-Entropy
- **Optimizer:** Adam

In [ ]:
from sklearn.neural_network import MLPClassifier

# Build the neural network
model = MLPClassifier(
    hidden_layer_sizes=(64,),    # One hidden layer with 64 neurons
    activation='relu',           # ReLU activation: max(0, x)
    solver='adam',               # Adam optimizer (adaptive learning rate)
    learning_rate_init=0.001,    # Starting learning rate
    batch_size=32,               # Process 32 samples at a time
    max_iter=100,                # 100 epochs (passes through training data)
    random_state=42
)

print("Neural Network Architecture:")
print("  Input Layer:    15 neurons (15 input features)")
print("  Hidden Layer:   64 neurons, ReLU activation")
print("  Output Layer:   1 neuron (binary: churn or not)")
print("  Loss Function:  Binary Cross-Entropy")
print("  Optimizer:      Adam")
print("  Learning Rate:  0.001")
print("  Batch Size:     32")

## Task 4: Training and Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

# Train the model
model.fit(X_train, y_train)

# Evaluate
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test,  model.predict(X_test))
print(f"Training Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Testing Accuracy:  {test_acc:.4f}  ({test_acc*100:.2f}%)")

# Plot training loss curve
plt.figure(figsize=(8, 4))
plt.plot(model.loss_curve_, color='steelblue', linewidth=2)
plt.title('Training Loss Curve')
plt.xlabel('Epoch (Training Iteration)')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/training_loss.png', dpi=120)
plt.show()
print("\nInterpretation: Loss decreases over epochs, showing the model is learning.")

In [ ]:
# Confusion matrix
y_pred = model.predict(X_test)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=['No Churn','Churn'], cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix - Baseline Model')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=120)
plt.show()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn','Churn']))
print("""
Interpretation:
- High accuracy (~98%) is due to class imbalance (98.5% of customers don't churn).
- The model predicts 'No Churn' well but struggles to detect actual churn cases.
- Low recall for 'Churn' class means some churning customers are missed.
- Business improvement: use class weighting or SMOTE for better churn detection.
""")

## Task 5: Hyperparameter Experimentation

We run 5 experiments, each changing one hyperparameter:

In [ ]:
experiments = {
    'Exp1 - Baseline':        dict(hidden_layer_sizes=(64,),    activation='relu', learning_rate_init=0.001, batch_size=32, max_iter=100),
    'Exp2 - Deeper Network':  dict(hidden_layer_sizes=(128,64), activation='relu', learning_rate_init=0.001, batch_size=32, max_iter=100),
    'Exp3 - High LR 0.01':   dict(hidden_layer_sizes=(64,),    activation='relu', learning_rate_init=0.01,  batch_size=32, max_iter=100),
    'Exp4 - Small Batch 16':  dict(hidden_layer_sizes=(64,),    activation='relu', learning_rate_init=0.001, batch_size=16, max_iter=100),
    'Exp5 - Tanh Activation': dict(hidden_layer_sizes=(64,),    activation='tanh', learning_rate_init=0.001, batch_size=32, max_iter=100),
}

rows = []
for name, cfg in experiments.items():
    m = MLPClassifier(solver='adam', random_state=42, **cfg)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test,  m.predict(X_test))
    rows.append({
        'Experiment': name,
        'Hidden Layers': str(cfg['hidden_layer_sizes']),
        'Activation': cfg['activation'],
        'Learning Rate': cfg['learning_rate_init'],
        'Batch Size': cfg['batch_size'],
        'Epochs': cfg['max_iter'],
        'Train Accuracy': round(tr, 4),
        'Test Accuracy':  round(te, 4)
    })
    print(f"{name}: Train={tr:.4f} | Test={te:.4f}")

comp_df = pd.DataFrame(rows)
comp_df.to_csv('results/model_comparison_table.csv', index=False)
print("\nSaved to results/model_comparison_table.csv")
comp_df

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(comp_df))
ax.bar(x - 0.2, comp_df['Train Accuracy'], 0.4, label='Train', color='steelblue')
ax.bar(x + 0.2, comp_df['Test Accuracy'],  0.4, label='Test',  color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(comp_df['Experiment'], rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Hyperparameter Experiment Results: Train vs Test Accuracy')
ax.legend()
ax.set_ylim(0.9, 1.01)
plt.tight_layout()
plt.savefig('results/model_comparison_table.png', dpi=120)
plt.show()

## Task 6: Final Reflection

### What role do weights and biases play in the model?
Every connection in a neural network has a **weight** - a number that controls how strongly one neuron influences the next. A **bias** is an extra value added so a neuron can activate even when all inputs are zero. During training, the network adjusts weights and biases using backpropagation to minimize prediction error. Think of weights as the "importance dial" for each feature.

### Why is an activation function required?
Without activation functions, stacking many layers would still produce only a linear equation (y = mx + b). Activation functions like **ReLU** (Rectified Linear Unit: `max(0, x)`) introduce **non-linearity**, letting the network learn complex patterns like "high payment delay AND low satisfaction score BOTH predict churn." Without it, a 10-layer network would be no better than a single straight line.

### What happens when the learning rate is too high or too low?
- **Too high (e.g., 1.0):** The model takes giant steps, overshoots the correct weights, and may oscillate wildly without ever converging to a good solution.
- **Too low (e.g., 0.00001):** The model takes tiny steps, learns extremely slowly, and may get stuck at a poor local minimum or need thousands of epochs.
- **Just right (e.g., 0.001 with Adam):** Smooth, steady improvement in loss over training.

### Did the model show signs of underfitting or overfitting?
The model shows **mild overfitting**: training accuracy (~99.9%) is slightly higher than test accuracy (~98%). The training accuracy is nearly perfect, meaning the model memorized the training data very well. The gap is small, so it is not severe overfitting. The larger issue is **class imbalance** - with only 31 churned customers in 2000, predicting "No Churn" for everyone gives 98.5% accuracy, so accuracy alone is misleading. Better metrics for this problem would be Precision, Recall, and F1-Score for the churn class.